# Занятие 2. Типы, преобразования, ветвления

**План занятия**

1. Тип принадлежит значению
2. Преобразования и их отказы
3. Истинность значений
4. Сравнения, `==` против `is`
5. Логические операторы и короткое замыкание
6. Ветвление, порядок проверок
7. Домашние задачи

**Теория:** `theory/02_Типы_и_ветвления.md`
Т. Гэддис, гл. 2 (с. 38 / PDF 63) и гл. 3 (с. 128 / PDF 153)

---

## 1. Тип принадлежит значению

Имя переменной типа не имеет. Одно имя может указывать на значения разных типов.

In [ ]:
value = 42
print(value, type(value))

value = "42"
print(value, type(value))

value = [42]
print(value, type(value))

Пять базовых типов:

In [ ]:
for item in [42, 3.14, "42", True, None]:
    print(f"{str(item):>6}  {type(item).__name__}")

`int` не ограничен размером машинного слова.

In [ ]:
big = 2 ** 1000
print(len(str(big)), "цифр")
print(big % 97)     # вычисляется точно

---

## 2. Преобразования

Числовые типы расширяются автоматически. Строка в этой цепочке не участвует.

In [ ]:
print(7 + 3.0)      # int расширился до float
print(True + 1)     # bool это подкласс int

try:
    print("5" + 5)
except TypeError as error:
    print("TypeError:", error)

`+` для строки означает склейку, для числа сложение. Выбрать за вас Python
отказывается.

Явное преобразование:

In [ ]:
print(int("2026"))
print(int(3.99))        # дробная часть отбрасывается без округления
print(int(-3.99))       # к нулю, а не вниз
print(float("3.14"))
print(str(42) + "!")

`int()` от строки требует, чтобы вся строка была записью целого числа.

In [ ]:
for raw in [" 42 ", "42.0", "forty", ""]:
    try:
        print(f"{raw!r:>8} -> {int(raw)}")
    except ValueError as error:
        print(f"{raw!r:>8} -> ValueError: {error}")

Отсюда правило для данных извне: проверяйте перед преобразованием
или ловите исключение (занятие 5).

### Точность float

In [ ]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)
print(f"{0.1 + 0.2:.20f}")

Двоичная дробь не представляет 0.1 точно. Свойство стандарта IEEE 754.

Правильное сравнение:

In [ ]:
import math

a, b = 0.1 + 0.2, 0.3
print(abs(a - b) < 1e-9)
print(math.isclose(a, b))

---

## 3. Истинность значений

В логическом контексте любое значение приводится к `True` или `False`.
Ложны «пустые» значения.

In [ ]:
for value in [0, 0.0, "", [], {}, set(), None, False, 1, "0", [0], " "]:
    print(f"{str(value):>8}  ->  {bool(value)}")

Обратите внимание на три последних: строка `"0"` истинна, список `[0]` истинен,
пробел истинен. Пусто означает пусто, а не «похоже на ноль».

In [ ]:
items = []

if items:
    print("непусто")
else:
    print("пусто")

# осторожно с числом 0
count = 0
if count:
    print("сработало")
else:
    print("ноль ложен, ветка пропущена")

Если ноль для вас осмысленное значение, проверяйте явно: `if count is not None`.

---

## 4. Сравнения

**Проверьте себя.** Что напечатает следующая ячейка? Сначала ответьте, потом запускайте.

In [ ]:
a = [1, 2]
b = [1, 2]

print(a == b)
print(a is b)

`==` сравнивает значения, `is` сравнивает тождество объектов.
`is` применяют к `None`, `True`, `False`, и больше почти ни к чему.

In [ ]:
value = None
print(value is None)         # так пишут
print(value == None)         # работает, так не пишут

Цепочки сравнений работают как в математике и вычисляются один раз:

In [ ]:
index, items = 3, [10, 20, 30, 40, 50]
print(0 <= index < len(items))

---

## 5. Логические операторы

Короткое замыкание: правый операнд не вычисляется, если результат ясен по левому.

In [ ]:
def loud(value, label):
    print(f"  вычислено: {label}")
    return value

print("A and:")
result = loud(False, "left") and loud(True, "right")

print("B or:")
result = loud(True, "left") or loud(True, "right")

Отсюда типовая защита от `IndexError`. Порядок здесь существенный.

In [ ]:
items = []
index = 0

print(index < len(items) and items[index] > 0)     # безопасно

try:
    print(items[index] > 0 and index < len(items))  # порядок обратный
except IndexError as error:
    print("IndexError:", error)

Сложное условие стоит назвать:

In [ ]:
value = 42

is_valid = 0 < value < 100 and value % 2 == 0
print(is_valid)

---

## 6. Ветвление

In [ ]:
def grade(score):
    if score >= 90:
        return "A"
    elif score >= 75:
        return "B"
    elif score >= 60:
        return "C"
    else:
        return "F"

for score in [95, 80, 65, 30]:
    print(score, grade(score))

### Порядок проверок

Классическая ошибка, которая не даёт сообщения.

In [ ]:
def grade_broken(score):
    if score >= 60:
        return "C"
    elif score >= 90:      # недостижимо
        return "A"
    else:
        return "F"

for score in [95, 80, 65, 30]:
    print(score, grade_broken(score))

Отличник получил C. Всё, что больше 90, уже больше 60, поэтому вторая ветка
не выполнится никогда. Программа работает и выдаёт неправильный результат.

Проверять надо от частного к общему.

### Условное выражение

In [ ]:
for n in range(5):
    print(n, "even" if n % 2 == 0 else "odd")

---

## 7. Собираем программу

Классификатор по нескольким порогам с валидацией входа.

In [ ]:
def classify(length):
    if not isinstance(length, int):
        return "invalid: not an integer"
    if length < 0:
        return "invalid: negative"
    if length == 0:
        return "empty"
    elif length < 100:
        return "short"
    elif length < 1000:
        return "medium"
    else:
        return "long"

for sample in [0, 42, 500, 5000, -1, 3.5]:
    print(f"{str(sample):>6}  {classify(sample)}")

Порядок веток здесь важен дважды: сначала отсеиваются некорректные входы,
потом пороги идут от меньшего к большему.

---

# Домашние задачи

Рассчитаны примерно на 30 минут.

### Задача 1. Трассировка (без запуска)

Не выполняя код, напишите, что он напечатает.

In [ ]:
# Мой ответ: ...

# x = "10"
# y = 3
# print(int(x) + y)
# print(x + str(y))
# print(x * y)
# print(bool(x), bool(""), bool(0), bool("0"))

### Задача 2. Минимум LeetCode

**LeetCode 2235 Add Two Integers** плюс письменный разбор хода мысли:
как поняли условие, какие крайние случаи придумали, каким было решение.

In [ ]:
def get_sum(num1, num2):
    # ваш код здесь
    pass

# print(get_sum(12, 5), get_sum(-10, 4))

### Задача 3. Классификатор с порогами

Дан балл от 0 до 100. Верните оценку по шкале: 85 и выше A, 70 и выше B,
55 и выше C, иначе F. Балл вне диапазона 0..100 считается ошибкой ввода.

Проверьте на границах: 85, 84, 0, 100, 101, −1.

In [ ]:
def grade(score):
    # ваш код здесь
    pass

for score in [85, 84, 0, 100, 101, -1]:
    print(score, grade(score))

### Задача 4. Найдите ошибки

В функции ниже три ошибки разных типов. Найдите, исправьте, для каждой запишите
в комментарии тип и причину.

In [ ]:
# def price_with_tax(raw_price, rate):
#     price = raw_price
#     if rate > 1:
#         rate = rate / 100
#     total = price * (1 + rate)
#     return "Total: " + total

### Задача 5. Эксперимент

Гипотеза: «`int()` округляет к ближайшему целому, поэтому `int(2.5)` даст 2,
`int(3.5)` даст 4, `int(-2.5)` даст −2».

Сначала запишите ожидаемый результат в комментарии, потом запускайте,
потом объясните расхождение.

In [ ]:
# Моя гипотеза: ...

print(int(2.5), int(3.5), int(-2.5), int(-3.5))
print(round(2.5), round(3.5), round(-2.5), round(-3.5))

# Что получилось и почему:

### Задача 6. Трек «алгоритмы» (по желанию)

1281 Subtract the Product and Sum of Digits, 412 Fizz Buzz, 9 Palindrome Number.

1281 использует `%` и `//`.

---

# Итоги

- Тип принадлежит значению, имя переменной типа не имеет.
- `int()` от строки требует, чтобы строка целиком была числом.
- `int(3.99)` даёт 3, отбрасывание вместо округления.
- Дробные числа не сравнивают на точное равенство.
- Ложны пустые значения. Строка `"0"` и список `[0]` истинны.
- `==` про значения, `is` про тождество. `is` используют с `None`.
- Короткое замыкание позволяет проверять границу перед доступом по индексу.
- Ветки проверяют от частного к общему, иначе часть из них недостижима.